In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Imports, Style and Paths ===

import json
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.image as mpimg
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D            # noqa: F401  registers "3d"

from scipy import ndimage
from sklearn.decomposition import PCA as _SkPCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

try:
    import umap
except ImportError:
    umap = None
    print("!! umap-learn not installed - UMAP sections will be skipped.")

try:
    import cv2 as _cv2
except ImportError:
    _cv2 = None
    print("!! cv2 not installed - Section 6 polygon rasterisation will be skipped.")

try:
    import tifffile
except ImportError:
    tifffile = None

try:
    from spectral.io import envi
except ImportError:
    envi = None

warnings.filterwarnings("ignore", category=FutureWarning)

_FW = 16 / 2.54
_FONT_SIZE = 10
_PREFERRED_FONTS = ["DejaVu Sans", "Helvetica", "Arial", "Nimbus Sans",
                    "TeX Gyre Heros", "Liberation Sans"]
_available_fonts = {f.name for f in fm.fontManager.ttflist}
_font = next((f for f in _PREFERRED_FONTS if f in _available_fonts), "DejaVu Sans")
plt.rcParams["font.family"] = _font
plt.rcParams["font.size"] = _FONT_SIZE
plt.rcParams["axes.titlesize"] = _FONT_SIZE + 1
plt.rcParams["axes.labelsize"] = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
plt.rcParams["xtick.labelsize"] = _FONT_SIZE - 1
plt.rcParams["ytick.labelsize"] = _FONT_SIZE - 1
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["legend.frameon"] = False
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"

REGION_ORDER = ("body", "cap", "label")
REGION_COLORS = {"body": "#FF0000", "cap": "#00FF00", "label": "#0000FF"}
CLASS_MAP = {"body": 0, "cap": 1, "label": 2}
CLASS_NAMES = {v: k for k, v in CLASS_MAP.items()}

RANDOM_SEED = 42

SW_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Spectral_Waste/KI_Sort_Preprocessing")
DATA_ROOT = SW_ROOT / "data"
RAW_SPECTRA_DIR = DATA_ROOT / "preprocessed_spectra_raw_20260901_071026"

TRUE_GT_RUN_ID = "20260820_115620_raw_labeled_v1"
ALIGN_RUN_ID = "20260819_065958_raw_labeled_HSI_alignment_table"
IARR_REF_CSV = DATA_ROOT / f"true_gt_iarr_reference_{TRUE_GT_RUN_ID}.csv"

OUT_DIR = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/08_Generalizability/"
               "20260901_Spectral_Waste_HSI_Investigation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"raw spectra   : {RAW_SPECTRA_DIR}")
print(f"iarr reference: {IARR_REF_CSV}")
print(f"output        : {OUT_DIR}")
print(f"run timestamp : {RUN_TS}")
print(f"umap available: {umap is not None}   cv2 available: {_cv2 is not None}")

In [ ]:
# === Cell 3: Loaders, Variants and Statistics ===

WL_MIN_NM, WL_MAX_NM = 900.0, 1700.0
N_BANDS = 224

SPECTRUM_SEP = ";"


def save_fig(fig, name):
    """Write a figure into the flat output folder."""
    p = OUT_DIR / f"{name}.png"
    fig.savefig(p)
    print(f"saved -> {p.name}")
    return p


def save_table(df, name):
    """Write a table into the flat output folder."""
    p = OUT_DIR / f"{name}.csv"
    df.to_csv(p, index=False)
    print(f"saved -> {p.name}")
    return p


def load_raw_spectra(folder=RAW_SPECTRA_DIR, regions=REGION_ORDER):
    """Return the (objects x bands) raw DN matrix, aligned metadata and wavelength axis.

    Files are '<cutout_id>_<region>' with two headerless columns 'wavelength_nm;value'.
    The first file's band grid defines the columns; files with a different grid are skipped.
    """
    meta_path = folder / "true_gt_spectra_meta.csv"
    if not meta_path.exists():
        raise FileNotFoundError(f"meta CSV absent: {meta_path}")

    meta = pd.read_csv(meta_path)
    meta = meta[meta["region"].isin(regions)].reset_index(drop=True)

    rows, wl, skipped, keep_rows = [], None, [], []
    for i, r in meta.iterrows():
        fpath = folder / f"{r['cutout_id']}_{r['region']}"
        if not fpath.exists():
            skipped.append(f"{fpath.name} (absent)")
            continue
        sp = pd.read_csv(fpath, sep=SPECTRUM_SEP, header=None,
                         names=["wavelength_nm", "value"])
        wl_this = sp["wavelength_nm"].to_numpy()
        if wl is None:
            wl = wl_this
        elif wl_this.shape != wl.shape or not np.allclose(wl, wl_this):
            skipped.append(f"{fpath.name} (band grid mismatch)")
            continue
        rows.append(sp["value"].to_numpy())
        keep_rows.append(i)

    if not rows:
        raise RuntimeError(f"no spectrum files read from {folder}")

    M = np.vstack(rows)
    idx = meta.loc[keep_rows].reset_index(drop=True)
    finite = ~np.isnan(M).any(axis=1)
    if (~finite).any():
        skipped.append(f"{int((~finite).sum())} spectra dropped for NaN bands")
    M, idx = M[finite], idx[finite].reset_index(drop=True)

    idx["cutout_id"] = idx["cutout_id"].astype(str)
    for c in ("session", "split"):
        if c not in idx.columns:
            idx[c] = ""
    idx["session"] = idx["session"].astype(str)

    if skipped:
        print(f"!! {len(skipped)} note(s): {skipped[:3]}")
    return M, idx, wl


def load_iarr_reference(wl, path=IARR_REF_CSV):
    """Dataset-wide per-band mean DN, interpolated onto the spectra's band grid."""
    if not path.exists():
        raise FileNotFoundError(f"IARR reference absent: {path}")
    ref = pd.read_csv(path)
    ref_wl = ref["wavelength_nm"].to_numpy()
    ref_dn = ref["mean_dn"].to_numpy()
    if ref_wl.shape == wl.shape and np.allclose(ref_wl, wl):
        return ref_dn
    return np.interp(wl, ref_wl, ref_dn)


def apply_iarr(M, ref_dn):
    """IARR normalisation: DN(b) / mean_DN(b) per band."""
    d = np.asarray(ref_dn, dtype=np.float64)
    d = np.where(np.abs(d) < 1e-8, np.nan, d)
    return M / d[None, :]


def apply_snv(M, ddof=0):
    """Per-spectrum standardisation over the retained band range."""
    mu = np.nanmean(M, axis=1, keepdims=True)
    sd = np.nanstd(M, axis=1, ddof=ddof, keepdims=True)
    return (M - mu) / (sd + 1e-9)


def apply_msc(M, reference=None):
    """Multiplicative scatter correction against the dataset mean spectrum."""
    if reference is None:
        reference = M.mean(axis=0)
    out = np.empty_like(M, dtype=np.float64)
    for i in range(M.shape[0]):
        b, a = np.polyfit(reference, M[i], 1)
        out[i] = (M[i] - a) / (b if abs(b) > 1e-12 else 1e-12)
    return out


def build_variants(M_raw, wl):
    """Build the raw, iarr, snv and msc variants from the raw matrix."""
    ref_dn = load_iarr_reference(wl)
    variants = {
        "raw": M_raw,
        "iarr": apply_iarr(M_raw, ref_dn),
        "snv": apply_snv(M_raw),
        "msc": apply_msc(M_raw),
    }
    for tag, M in variants.items():
        n_bad = int(np.isnan(M).any(axis=1).sum())
        if n_bad:
            print(f"!! [{tag}] {n_bad} spectra contain NaN after transform")
    return variants


def add_confound_columns(idx, M_raw):
    """Add mean raw DN (mean_level) and its quartile bin (mean_level_bin).

    mean_level always comes from the raw matrix, so it is independent of clusters
    fit in a normalised space.
    """
    idx = idx.copy()
    idx["mean_level"] = M_raw.mean(axis=1)
    try:
        idx["mean_level_bin"] = pd.qcut(
            idx["mean_level"], 4, labels=["Q1(low)", "Q2", "Q3", "Q4(high)"],
            duplicates="drop").astype(str)
    except ValueError:
        idx["mean_level_bin"] = "all"
    return idx


def hopkins_statistic(X, seed):
    """Hopkins statistic: about 0.5 for uniform data, towards 1 for clustered data."""
    rng = np.random.RandomState(seed)
    n, d = X.shape
    n_samples = min(n - 1, max(5, n // 5))
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    take = rng.choice(n, n_samples, replace=False)
    d_real, _ = nn.kneighbors(X[take], n_neighbors=2)
    u_real = d_real[:, 1]
    rand_pts = rng.uniform(X.min(axis=0), X.max(axis=0), size=(n_samples, d))
    d_rand, _ = nn.kneighbors(rand_pts, n_neighbors=1)
    u_rand = d_rand[:, 0]
    denom = u_rand.sum() + u_real.sum()
    return u_rand.sum() / denom if denom > 0 else np.nan


def eta_squared(values, groups):
    """One-way ANOVA eta squared of values grouped by a label (used on PCA scores)."""
    df_ = pd.DataFrame({"v": np.asarray(values, dtype=float), "g": np.asarray(groups)})
    grand = df_["v"].mean()
    ss_tot = ((df_["v"] - grand) ** 2).sum()
    if ss_tot <= 0:
        return np.nan
    ss_between = df_.groupby("g")["v"].apply(
        lambda s: len(s) * (s.mean() - grand) ** 2).sum()
    return float(ss_between / ss_tot)


def standardise(X):
    return (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)


def cluster_profile(X, k_max=6, random_state=RANDOM_SEED, hopkins_repeats=5):
    """Hopkins statistic plus a KMeans silhouette sweep over k=2..k_max.

    Returns (hopkins, sweep, k at max silhouette, max silhouette, labels at that k).
    """
    n = X.shape[0]
    Xs = standardise(X)
    H = float(np.mean([hopkins_statistic(Xs, seed=s) for s in range(hopkins_repeats)]))
    sweep, labels_by_k = {}, {}
    for k in range(2, k_max + 1):
        if n <= k:
            continue
        km = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(Xs)
        sweep[k] = float(silhouette_score(Xs, km.labels_))
        labels_by_k[k] = km.labels_
    if not sweep:
        return H, {}, None, np.nan, None
    k_arg = max(sweep, key=sweep.get)
    return H, sweep, k_arg, sweep[k_arg], labels_by_k[k_arg]


def pca_scores(M, n_pcs, random_state=RANDOM_SEED):
    n_eff = int(min(n_pcs, M.shape[0] - 1, M.shape[1]))
    pca = _SkPCA(n_components=n_eff, random_state=random_state)
    scores = pca.fit_transform(M - M.mean(axis=0, keepdims=True))
    return scores, pca.explained_variance_ratio_, pca.components_


def kmeans_labels(X, k, random_state=RANDOM_SEED):
    return KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(
        standardise(X)).labels_


M_RAW, IDX, WL = load_raw_spectra()
IDX = add_confound_columns(IDX, M_RAW)
VARIANTS = build_variants(M_RAW, WL)
VARIANT_ORDER = ["raw", "iarr", "snv", "msc"]

print(f"objects: {M_RAW.shape[0]:,}   bands: {M_RAW.shape[1]}   "
      f"wl: {WL.min():.0f}-{WL.max():.0f} nm")
print("region counts:", IDX["region"].value_counts().to_dict())
print("sessions:", IDX["session"].nunique())
print("variants:", VARIANT_ORDER)

In [ ]:
# === Cell 4: PCA Loading Spectra ===

N_PCS_LOADING = 4

loadings_by_variant = {}
fig, axes = plt.subplots(len(VARIANT_ORDER), 1,
                         figsize=(_FW * 1.5, _FW * 0.45 * len(VARIANT_ORDER)),
                         sharex=True)

for row_i, tag in enumerate(VARIANT_ORDER):
    M = VARIANTS[tag]
    scores, evr, comps = pca_scores(M, N_PCS_LOADING)
    loadings_by_variant[tag] = {"evr": evr, "loadings": comps}

    ax = axes[row_i]
    for i in range(comps.shape[0]):
        ax.plot(WL, comps[i], lw=1.1, label=f"PC{i + 1} ({evr[i] * 100:.1f}%)")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_ylabel(f"{tag}\nloading")
    ax.legend(loc="upper right", ncol=2)

axes[0].set_title("PCA Loading Spectra by Preprocessing Variant")
axes[-1].set_xlabel("Wavelength (nm)")
fig.tight_layout()
plt.show()
save_fig(fig, "fig01_pca_loading_spectra_all_variants")

rows = []
for tag, d in loadings_by_variant.items():
    for i in range(d["loadings"].shape[0]):
        L = d["loadings"][i]
        rows.append({
            "variant": tag, "PC": i + 1,
            "explained_var_pct": round(float(d["evr"][i] * 100), 2),
            "wl_at_max_pos_loading_nm": round(float(WL[np.argmax(L)]), 1),
            "wl_at_max_neg_loading_nm": round(float(WL[np.argmin(L)]), 1),
            "loading_sign_flips": int(np.sum(np.diff(np.sign(L)) != 0)),
        })
loading_summary = pd.DataFrame(rows)
print(loading_summary.to_string(index=False))
save_table(loading_summary, "tbl01_pca_loading_summary")

In [ ]:
# === Cell 5: Edge-Band Trim Sensitivity ===

TRIM_FRACTION = 0.05          # removed from each end
N_PCS_TRIM = 4

n_bands = WL.shape[0]
trim_n = int(round(n_bands * TRIM_FRACTION))
keep_mask = np.zeros(n_bands, dtype=bool)
keep_mask[trim_n: n_bands - trim_n] = True
WL_TRIM = WL[keep_mask]

print(f"full  : {n_bands} bands  {WL[0]:.0f}-{WL[-1]:.0f} nm")
print(f"trim  : {WL_TRIM.shape[0]} bands  {WL_TRIM[0]:.0f}-{WL_TRIM[-1]:.0f} nm  "
      f"(removed {WL[0]:.0f}-{WL[trim_n - 1]:.0f} and "
      f"{WL[n_bands - trim_n]:.0f}-{WL[-1]:.0f} nm)")

trim_rows = []
fig, axes = plt.subplots(len(VARIANT_ORDER), 2,
                         figsize=(_FW * 1.9, _FW * 0.45 * len(VARIANT_ORDER)),
                         squeeze=False)

for row_i, tag in enumerate(VARIANT_ORDER):
    M_full = VARIANTS[tag]
    M_trim = M_full[:, keep_mask]
    _, evr_f, comp_f = pca_scores(M_full, N_PCS_TRIM)
    _, evr_t, comp_t = pca_scores(M_trim, N_PCS_TRIM)

    ax = axes[row_i][0]
    for i in range(min(2, comp_f.shape[0])):
        ax.plot(WL, comp_f[i], lw=1.1, label=f"PC{i + 1} ({evr_f[i] * 100:.1f}%)")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_ylabel(f"{tag}\nloading")
    ax.legend(loc="upper right")
    if row_i == 0:
        ax.set_title("Full Range")

    ax = axes[row_i][1]
    for i in range(min(2, comp_t.shape[0])):
        ax.plot(WL_TRIM, comp_t[i], lw=1.1, label=f"PC{i + 1} ({evr_t[i] * 100:.1f}%)")
    ax.axhline(0, color="black", lw=0.5)
    ax.legend(loc="upper right")
    if row_i == 0:
        ax.set_title(f"Trimmed {TRIM_FRACTION * 200:.0f}%")

    for i in range(min(N_PCS_TRIM, comp_f.shape[0], comp_t.shape[0])):
        Lf, Lt = comp_f[i], comp_t[i]
        trim_rows.append({
            "variant": tag, "PC": i + 1,
            "evr_full_pct": round(float(evr_f[i] * 100), 2),
            "evr_trim_pct": round(float(evr_t[i] * 100), 2),
            "evr_delta_pct": round(float((evr_t[i] - evr_f[i]) * 100), 2),
            "wl_max_pos_full_nm": round(float(WL[np.argmax(Lf)]), 1),
            "wl_max_pos_trim_nm": round(float(WL_TRIM[np.argmax(Lt)]), 1),
            "wl_max_neg_full_nm": round(float(WL[np.argmin(Lf)]), 1),
            "wl_max_neg_trim_nm": round(float(WL_TRIM[np.argmin(Lt)]), 1),
            "sign_flips_full": int(np.sum(np.diff(np.sign(Lf)) != 0)),
            "sign_flips_trim": int(np.sum(np.diff(np.sign(Lt)) != 0)),
        })

axes[-1][0].set_xlabel("Wavelength (nm)")
axes[-1][1].set_xlabel("Wavelength (nm)")
fig.suptitle("Loading Spectra, Full vs. Edge-Trimmed Band Range", y=1.005)
fig.tight_layout()
plt.show()
save_fig(fig, "fig02_edge_band_trim_loadings")

trim_summary = pd.DataFrame(trim_rows)
pd.set_option("display.width", 220)
print(trim_summary.to_string(index=False))
save_table(trim_summary, "tbl02_edge_band_trim_sensitivity")

In [ ]:
# === Cell 6: Confound Eta-Squared per PC ===

# eta squared of session and mean-level bin on the leading PC scores of each variant.
N_PCS_ETA = 20
N_PCS_ETA_REPORT = 5

eta_rows = []
for tag in VARIANT_ORDER:
    M = VARIANTS[tag]
    scores, evr, _ = pca_scores(M, N_PCS_ETA)
    for i in range(min(N_PCS_ETA_REPORT, scores.shape[1])):
        eta_rows.append({
            "variant": tag, "PC": i + 1,
            "explained_var_pct": round(float(evr[i] * 100), 2),
            "eta2_session": round(eta_squared(scores[:, i], IDX["session"]), 3),
            "eta2_mean_level_bin": round(
                eta_squared(scores[:, i], IDX["mean_level_bin"]), 3),
        })
    cum3 = float(evr[:min(3, len(evr))].sum() * 100)
    print(f"[{tag}] cumulative variance PC1-3 = {cum3:.1f}%   "
          f"all {len(evr)} PCs = {evr.sum() * 100:.1f}%")

eta_df = pd.DataFrame(eta_rows)
print("\n" + eta_df.to_string(index=False))
save_table(eta_df, "tbl03_confound_eta2_all_variants")

fig, ax = plt.subplots(figsize=(_FW * 1.2, _FW * 0.55))
pc1 = eta_df[eta_df["PC"] == 1]
x = np.arange(len(pc1))
w = 0.38
ax.bar(x - w / 2, pc1["eta2_mean_level_bin"], w, label="mean level bin",
       color="#5A56A1")
ax.bar(x + w / 2, pc1["eta2_session"], w, label="session", color="#88BEE7")
ax.set_xticks(x)
ax.set_xticklabels(pc1["variant"])
ax.set_ylabel("eta squared")
ax.set_ylim(0, 1.0)
ax.set_title("PC1 Variance Attributable to Confounds")
ax.legend(loc="upper right")
for xi, (mb, se) in enumerate(zip(pc1["eta2_mean_level_bin"], pc1["eta2_session"])):
    ax.text(xi - w / 2, mb + 0.02, f"{mb:.2f}", ha="center", fontsize=_FONT_SIZE - 2)
    ax.text(xi + w / 2, se + 0.02, f"{se:.2f}", ha="center", fontsize=_FONT_SIZE - 2)
fig.tight_layout()
plt.show()
save_fig(fig, "fig03_pc1_confound_eta2")

In [ ]:
# === Cell 7: t-SNE Hyperparameter Grid ===

TSNE_GRID_VARIANTS = ["snv", "msc"]
TSNE_N_PCS_IN = 20
PERPLEXITY_GRID = (5, 15, 30, 50, 100)
EARLY_EXAG_GRID = (4, 12, 24)

tsne_grid_rows, tsne_grid_coords = [], []

for tag in TSNE_GRID_VARIANTS:
    M = VARIANTS[tag]
    scores, evr, _ = pca_scores(M, TSNE_N_PCS_IN)
    n_in = scores.shape[1]
    print(f"[{tag}] PCA -> {n_in} PCs, cumulative variance {evr.sum() * 100:.1f}%")

    fig, axes = plt.subplots(len(PERPLEXITY_GRID), len(EARLY_EXAG_GRID),
                             figsize=(_FW * 0.55 * len(EARLY_EXAG_GRID),
                                      _FW * 0.55 * len(PERPLEXITY_GRID)),
                             squeeze=False)

    for ri, perp in enumerate(PERPLEXITY_GRID):
        perp_eff = float(min(perp, max(5.0, (M.shape[0] - 1) / 3.0)))
        for ci, exag in enumerate(EARLY_EXAG_GRID):
            emb = TSNE(n_components=2, perplexity=perp_eff, early_exaggeration=exag,
                       init="pca", random_state=RANDOM_SEED).fit_transform(scores)

            Xs = standardise(emb)
            sil_region = float(silhouette_score(Xs, IDX["region"]))
            H = float(np.mean([hopkins_statistic(Xs, seed=s) for s in range(3)]))

            ax = axes[ri][ci]
            for region, g in IDX.groupby("region"):
                m = g.index.to_numpy()
                ax.scatter(emb[m, 0], emb[m, 1], s=3, alpha=0.55,
                           color=REGION_COLORS[region])
            ax.set_xticks([]); ax.set_yticks([])
            ax.text(0.02, 0.98, f"sil={sil_region:.3f}", va="top", ha="left",
                    transform=ax.transAxes, fontsize=_FONT_SIZE - 3)
            if ri == 0:
                ax.set_title(f"early_exag={exag}")
            if ci == 0:
                ax.set_ylabel(f"perplexity={perp}")

            tsne_grid_rows.append({
                "variant": tag, "perplexity": perp, "perplexity_effective": perp_eff,
                "early_exaggeration": exag, "n_pcs_in": n_in,
                "silhouette_region": round(sil_region, 3), "hopkins": round(H, 3),
            })
            cdf = IDX[["cutout_id", "region", "session"]].copy()
            cdf["variant"] = tag
            cdf["perplexity"] = perp
            cdf["early_exaggeration"] = exag
            cdf["tsne_x"], cdf["tsne_y"] = emb[:, 0], emb[:, 1]
            tsne_grid_coords.append(cdf)

    handles = [Line2D([0], [0], marker="o", linestyle="", markersize=6,
                      color=REGION_COLORS[r], label=r) for r in REGION_ORDER]
    fig.legend(handles=handles, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(f"t-SNE Stability Grid - variant={tag} "
                 f"(silhouette against true region label)", y=1.05)
    fig.tight_layout()
    plt.show()
    save_fig(fig, f"fig04_tsne_hp_grid_{tag}")

tsne_grid_df = pd.DataFrame(tsne_grid_rows)
print(tsne_grid_df.to_string(index=False))
save_table(tsne_grid_df, "tbl04_tsne_hp_grid_metrics")
save_table(pd.concat(tsne_grid_coords, ignore_index=True), "tbl05_tsne_hp_grid_coords")

In [ ]:
# === Cell 8: UMAP Hyperparameter Grid ===

UMAP_GRID_VARIANTS = ["snv", "msc"]
UMAP_N_PCS_IN = 20
N_NEIGHBORS_GRID = (15, 30, 50, 100)
MIN_DIST_GRID = (0.0, 0.1, 0.3, 0.5)
UMAP_METRIC = "euclidean"

umap_grid_rows, umap_grid_coords = [], []

if umap is None:
    print("!! umap-learn not installed - skipping.")
else:
    for tag in UMAP_GRID_VARIANTS:
        M = VARIANTS[tag]
        scores, evr, _ = pca_scores(M, UMAP_N_PCS_IN)
        n_in = scores.shape[1]
        print(f"[{tag}] PCA -> {n_in} PCs, cumulative variance {evr.sum() * 100:.1f}%")

        fig, axes = plt.subplots(len(N_NEIGHBORS_GRID), len(MIN_DIST_GRID),
                                 figsize=(_FW * 0.55 * len(MIN_DIST_GRID),
                                          _FW * 0.55 * len(N_NEIGHBORS_GRID)),
                                 squeeze=False)

        for ri, n_nb in enumerate(N_NEIGHBORS_GRID):
            n_nb_eff = int(min(n_nb, M.shape[0] - 1))
            for ci, md in enumerate(MIN_DIST_GRID):
                emb = umap.UMAP(n_neighbors=n_nb_eff, min_dist=md, n_components=2,
                                metric=UMAP_METRIC,
                                random_state=RANDOM_SEED).fit_transform(scores)

                Xs = standardise(emb)
                sil_region = float(silhouette_score(Xs, IDX["region"]))
                H = float(np.mean([hopkins_statistic(Xs, seed=s) for s in range(3)]))

                ax = axes[ri][ci]
                for region, g in IDX.groupby("region"):
                    m = g.index.to_numpy()
                    ax.scatter(emb[m, 0], emb[m, 1], s=3, alpha=0.55,
                               color=REGION_COLORS[region])
                ax.set_xticks([]); ax.set_yticks([])
                ax.text(0.02, 0.98, f"sil={sil_region:.3f}", va="top", ha="left",
                        transform=ax.transAxes, fontsize=_FONT_SIZE - 3)
                if ri == 0:
                    ax.set_title(f"min_dist={md:g}")
                if ci == 0:
                    ax.set_ylabel(f"n_neighbors={n_nb}")

                umap_grid_rows.append({
                    "variant": tag, "n_neighbors": n_nb,
                    "n_neighbors_effective": n_nb_eff, "min_dist": md,
                    "n_pcs_in": n_in, "silhouette_region": round(sil_region, 3),
                    "hopkins": round(H, 3),
                })
                cdf = IDX[["cutout_id", "region", "session"]].copy()
                cdf["variant"] = tag
                cdf["n_neighbors"] = n_nb
                cdf["min_dist"] = md
                cdf["umap_x"], cdf["umap_y"] = emb[:, 0], emb[:, 1]
                umap_grid_coords.append(cdf)

        handles = [Line2D([0], [0], marker="o", linestyle="", markersize=6,
                          color=REGION_COLORS[r], label=r) for r in REGION_ORDER]
        fig.legend(handles=handles, loc="upper center", ncol=3,
                   bbox_to_anchor=(0.5, 1.02))
        fig.suptitle(f"UMAP Stability Grid - variant={tag} "
                     f"(silhouette against true region label)", y=1.05)
        fig.tight_layout()
        plt.show()
        save_fig(fig, f"fig05_umap_hp_grid_{tag}")

    umap_grid_df = pd.DataFrame(umap_grid_rows)
    print(umap_grid_df.to_string(index=False))
    save_table(umap_grid_df, "tbl06_umap_hp_grid_metrics")
    save_table(pd.concat(umap_grid_coords, ignore_index=True),
               "tbl07_umap_hp_grid_coords")

In [ ]:
# === Cell 9: PCA, t-SNE and UMAP Embeddings ===
# t-SNE and UMAP settings are chosen from the hyperparameter grid cells.
# Silhouettes use the true metadata labels, not KMeans clusters.

EMBED_N_PCS = 20
TSNE_PERPLEXITY = 30
TSNE_EARLY_EXAG = 12
UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.3
LABEL_VARS = ("region", "session", "mean_level_bin")

EMBEDDINGS = {}
embed_meta_rows, embed_sil_rows = [], []

for tag in VARIANT_ORDER:
    M = VARIANTS[tag]
    scores, evr, _ = pca_scores(M, EMBED_N_PCS)
    n_in = scores.shape[1]

    perp_eff = float(min(TSNE_PERPLEXITY, max(5.0, (M.shape[0] - 1) / 3.0)))
    tsne_emb = TSNE(n_components=2, perplexity=perp_eff,
                    early_exaggeration=TSNE_EARLY_EXAG, init="pca",
                    random_state=RANDOM_SEED).fit_transform(scores)

    if umap is not None:
        nb_eff = int(min(UMAP_N_NEIGHBORS, M.shape[0] - 1))
        umap_emb = umap.UMAP(n_neighbors=nb_eff, min_dist=UMAP_MIN_DIST,
                             n_components=2, metric="euclidean",
                             random_state=RANDOM_SEED).fit_transform(scores)
    else:
        nb_eff = None
        umap_emb = None

    EMBEDDINGS[tag] = {"scores": scores, "evr": evr, "tsne": tsne_emb,
                       "umap": umap_emb, "n_pcs_in": n_in}

    cum3 = float(evr[:min(3, len(evr))].sum() * 100)
    embed_meta_rows.append({
        "variant": tag, "n_objects": M.shape[0], "n_bands": M.shape[1],
        "n_pcs": n_in, "cum_var_pc1_3_pct": round(cum3, 2),
        "cum_var_all_pcs_pct": round(float(evr.sum() * 100), 2),
        "tsne_perplexity": perp_eff, "tsne_early_exaggeration": TSNE_EARLY_EXAG,
        "umap_n_neighbors": nb_eff, "umap_min_dist": UMAP_MIN_DIST,
        "random_seed": RANDOM_SEED,
    })

    spaces = {"pca3": scores[:, :min(3, n_in)], "tsne": tsne_emb}
    if umap_emb is not None:
        spaces["umap"] = umap_emb
    for space_name, X in spaces.items():
        Xs = standardise(X)
        row = {"variant": tag, "space": space_name,
               "hopkins": round(float(np.mean(
                   [hopkins_statistic(Xs, seed=s) for s in range(5)])), 3)}
        for var in LABEL_VARS:
            groups = IDX[var].astype(str)
            sil = (float(silhouette_score(Xs, groups))
                   if 2 <= groups.nunique() < len(groups) else np.nan)
            row[f"silhouette_{var}"] = round(sil, 3) if not np.isnan(sil) else np.nan
        embed_sil_rows.append(row)

    print(f"[{tag}] PCs={n_in}  cum var PC1-3={cum3:.1f}%  "
          f"tsne perplexity={perp_eff:g}  umap n_neighbors={nb_eff}")

embed_meta = pd.DataFrame(embed_meta_rows)
embed_sil = pd.DataFrame(embed_sil_rows)
print("\n=== Silhouette against true metadata labels (no KMeans) ===")
print(embed_sil.to_string(index=False))
save_table(embed_meta, "tbl08_embedding_run_metadata")
save_table(embed_sil, "tbl09_embedding_true_label_silhouette")

In [ ]:
# === Cell 10: Embedding Confound Grids ===

def _paint_by_region(ax, xy):
    for region, g in IDX.groupby("region"):
        m = g.index.to_numpy()
        ax.scatter(xy[m, 0], xy[m, 1], s=5, alpha=0.6, color=REGION_COLORS[region])


def _paint_by_session(ax, xy):
    pal = plt.get_cmap("tab20")
    for i, sess in enumerate(sorted(IDX["session"].unique())):
        m = IDX.index[IDX["session"] == sess].to_numpy()
        ax.scatter(xy[m, 0], xy[m, 1], s=5, alpha=0.6, color=pal(i % 20))


def _paint_by_level(ax, xy):
    return ax.scatter(xy[:, 0], xy[:, 1], s=5, alpha=0.6,
                      c=IDX["mean_level"], cmap="viridis")


for space_name, axis_label in (("tsne", "t-SNE"), ("umap", "UMAP")):
    if space_name == "umap" and umap is None:
        continue
    fig, axes = plt.subplots(len(VARIANT_ORDER), 3,
                             figsize=(_FW * 1.75, _FW * 0.52 * len(VARIANT_ORDER)),
                             squeeze=False)
    for r, tag in enumerate(VARIANT_ORDER):
        xy = EMBEDDINGS[tag][space_name]
        if xy is None:
            continue

        ax = axes[r][0]
        _paint_by_region(ax, xy)
        ax.set_ylabel(f"{tag}\n{axis_label} 2")
        if r == 0:
            ax.set_title("Region")

        ax = axes[r][1]
        _paint_by_session(ax, xy)
        if r == 0:
            ax.set_title("Session")
        ax.text(0.02, 0.98, f"{IDX['session'].nunique()} sessions", va="top",
                ha="left", transform=ax.transAxes, fontsize=_FONT_SIZE - 3)

        ax = axes[r][2]
        sc = _paint_by_level(ax, xy)
        if r == 0:
            ax.set_title("Mean Level")
        cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
        cb.ax.tick_params(labelsize=_FONT_SIZE - 4)

        for c in range(3):
            axes[r][c].set_xticks([]); axes[r][c].set_yticks([])
        axes[r][0].set_ylabel(f"{tag}\n{axis_label} 2")

    handles = [Line2D([0], [0], marker="o", linestyle="", markersize=6,
                      color=REGION_COLORS[k], label=k) for k in REGION_ORDER]
    fig.legend(handles=handles, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.01))
    fig.suptitle(f"{axis_label} Embedding by Preprocessing Variant", y=1.03)
    fig.tight_layout()
    plt.show()
    save_fig(fig, f"fig06_{space_name}_confound_grid_all_variants")

fig = plt.figure(figsize=(_FW * 1.75, _FW * 0.60 * len(VARIANT_ORDER)))
for r, tag in enumerate(VARIANT_ORDER):
    d = EMBEDDINGS[tag]
    scores, evr = d["scores"], d["evr"]
    for c, title in enumerate(("Region", "Session", "Mean Level")):
        ax = fig.add_subplot(len(VARIANT_ORDER), 3, r * 3 + c + 1, projection="3d")
        if c == 0:
            for region, g in IDX.groupby("region"):
                m = g.index.to_numpy()
                ax.scatter(scores[m, 0], scores[m, 1], scores[m, 2], s=5, alpha=0.6,
                           color=REGION_COLORS[region], depthshade=False)
        elif c == 1:
            pal = plt.get_cmap("tab20")
            for i, sess in enumerate(sorted(IDX["session"].unique())):
                m = IDX.index[IDX["session"] == sess].to_numpy()
                ax.scatter(scores[m, 0], scores[m, 1], scores[m, 2], s=5, alpha=0.6,
                           color=pal(i % 20), depthshade=False)
        else:
            sc = ax.scatter(scores[:, 0], scores[:, 1], scores[:, 2], s=5, alpha=0.6,
                            c=IDX["mean_level"], cmap="viridis", depthshade=False)
            cb = fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.10)
            cb.ax.tick_params(labelsize=_FONT_SIZE - 4)

        ax.set_xlabel(f"PC1 ({evr[0] * 100:.1f}%)", fontsize=_FONT_SIZE - 2)
        ax.set_ylabel(f"PC2 ({evr[1] * 100:.1f}%)", fontsize=_FONT_SIZE - 2)
        ax.set_zlabel(f"PC3 ({evr[2] * 100:.1f}%)", fontsize=_FONT_SIZE - 2)
        ax.tick_params(labelsize=_FONT_SIZE - 4)
        if r == 0:
            ax.set_title(title, pad=12)
        if c == 0:
            ax.text2D(-0.20, 0.5, f"{tag}\ncum PC1-3 "
                      f"{evr[:3].sum() * 100:.1f}%", transform=ax.transAxes,
                      rotation=90, va="center", ha="center", fontsize=_FONT_SIZE)

fig.suptitle("Principal Component Scores, First Three Components", y=1.005)
fig.tight_layout()
plt.show()
save_fig(fig, "fig07_pca3d_confound_grid_all_variants")

coord_rows = []
for tag in VARIANT_ORDER:
    d = EMBEDDINGS[tag]
    cdf = IDX[["cutout_id", "region", "session", "split", "mean_level",
               "mean_level_bin"]].copy()
    cdf["variant"] = tag
    for i in range(min(3, d["scores"].shape[1])):
        cdf[f"PC{i + 1}"] = d["scores"][:, i]
    cdf["tsne_x"], cdf["tsne_y"] = d["tsne"][:, 0], d["tsne"][:, 1]
    if d["umap"] is not None:
        cdf["umap_x"], cdf["umap_y"] = d["umap"][:, 0], d["umap"][:, 1]
    coord_rows.append(cdf)
save_table(pd.concat(coord_rows, ignore_index=True), "tbl10_embedding_coordinates")

In [ ]:
# === Cell 11: Per-Region Cluster Tendency ===

REGION_VARIANTS = ["raw", "iarr", "snv", "msc"]
REGION_N_PCS = 5
REGION_K_MAX = 6

region_rows = []
for tag in REGION_VARIANTS:
    M = VARIANTS[tag]
    for region in REGION_ORDER:
        mask = (IDX["region"] == region).to_numpy()
        M_r = M[mask]
        if M_r.shape[0] < 10:
            continue
        scores, evr, _ = pca_scores(M_r, REGION_N_PCS)
        H, sweep, k_arg, sil_arg, labels = cluster_profile(
            scores[:, :min(3, scores.shape[1])], k_max=REGION_K_MAX)

        row = {"variant": tag, "region": region, "n": int(M_r.shape[0]),
               "cum_var_pc1_2_pct": round(float(evr[:2].sum() * 100), 1),
               "hopkins": round(H, 3), "k_argmax_silhouette": k_arg,
               "silhouette_at_argmax": round(sil_arg, 3) if sweep else np.nan}
        for k in range(2, REGION_K_MAX + 1):
            row[f"silhouette_k{k}"] = round(sweep[k], 3) if k in sweep else np.nan
        if labels is not None:
            row["cluster_sizes_at_argmax"] = "/".join(
                str(s) for s in np.bincount(labels).tolist())
        region_rows.append(row)

region_df = pd.DataFrame(region_rows)
print(region_df[["variant", "region", "n", "cum_var_pc1_2_pct", "hopkins",
                 "k_argmax_silhouette", "silhouette_at_argmax",
                 "cluster_sizes_at_argmax"]].to_string(index=False))
save_table(region_df, "tbl11_per_region_cluster_tendency")

fig, axes = plt.subplots(1, len(REGION_ORDER),
                         figsize=(_FW * 0.62 * len(REGION_ORDER), _FW * 0.55),
                         sharey=True)
ks = list(range(2, REGION_K_MAX + 1))
for ax, region in zip(axes, REGION_ORDER):
    for tag in REGION_VARIANTS:
        sub = region_df[(region_df["variant"] == tag) &
                        (region_df["region"] == region)]
        if sub.empty:
            continue
        vals = [sub[f"silhouette_k{k}"].iloc[0] for k in ks]
        ax.plot(ks, vals, marker="o", ms=4, label=tag)
    ax.set_title(region)
    ax.set_xlabel("k")
    ax.set_xticks(ks)
axes[0].set_ylabel("silhouette (KMeans labels)")
axes[0].legend(loc="best")
fig.suptitle("Per-Region Silhouette Sweep", y=1.02)
fig.tight_layout()
plt.show()
save_fig(fig, "fig08_per_region_silhouette_sweep")

In [ ]:
# === Cell 12: Per-Region k=2 Clustering Agreement ===

CLUSTER_VARIANTS = ["raw", "snv", "msc"]
CLUSTER_K = 2
CLUSTER_N_PCS = 5
# KMeans (k=2) on PC1-3 per region and variant; ARI between variants; Cohen's d of raw
# mean_level between the two clusters.

REGION_CLUSTERS = {}          # (region, variant) -> DataFrame[cutout_id, cluster]
fixed_rows, ari_rows, brightness_rows = [], [], []

for region in REGION_ORDER:
    mask = (IDX["region"] == region).to_numpy()
    idx_r = IDX[mask].reset_index(drop=True)

    for tag in CLUSTER_VARIANTS:
        M_r = VARIANTS[tag][mask]
        scores, _, _ = pca_scores(M_r, CLUSTER_N_PCS)
        X = scores[:, :min(3, scores.shape[1])]
        labels = kmeans_labels(X, CLUSTER_K)
        sil = float(silhouette_score(standardise(X), labels))

        REGION_CLUSTERS[(region, tag)] = pd.DataFrame({
            "cutout_id": idx_r["cutout_id"].to_numpy(), "cluster": labels})

        fixed_rows.append({
            "region": region, "variant": tag, "n": int(M_r.shape[0]),
            "silhouette_k2": round(sil, 3),
            "cluster_sizes": "/".join(str(s) for s in np.bincount(labels).tolist()),
        })

        merged_b = REGION_CLUSTERS[(region, tag)].merge(
            idx_r[["cutout_id", "mean_level"]], on="cutout_id")
        desc = merged_b.groupby("cluster")["mean_level"].agg(["count", "mean", "std"])
        pooled_sd = merged_b["mean_level"].std()
        d = (abs(desc["mean"].iloc[0] - desc["mean"].iloc[1]) / pooled_sd
             if len(desc) == 2 and pooled_sd > 0 else np.nan)
        brightness_rows.append({
            "region": region, "variant": tag,
            "cluster0_n": int(desc["count"].iloc[0]),
            "cluster0_mean_level": round(float(desc["mean"].iloc[0]), 1),
            "cluster1_n": int(desc["count"].iloc[1]) if len(desc) == 2 else np.nan,
            "cluster1_mean_level": round(float(desc["mean"].iloc[1]), 1)
            if len(desc) == 2 else np.nan,
            "cohens_d_vs_raw_mean_level": round(float(d), 2) if not np.isnan(d) else np.nan,
        })

    for i in range(len(CLUSTER_VARIANTS)):
        for j in range(i + 1, len(CLUSTER_VARIANTS)):
            t1, t2 = CLUSTER_VARIANTS[i], CLUSTER_VARIANTS[j]
            m = REGION_CLUSTERS[(region, t1)].merge(
                REGION_CLUSTERS[(region, t2)], on="cutout_id",
                suffixes=(f"_{t1}", f"_{t2}"))
            ari = adjusted_rand_score(m[f"cluster_{t1}"], m[f"cluster_{t2}"])
            ari_rows.append({"region": region, "variant_a": t1, "variant_b": t2,
                             "n_matched": len(m), "ari": round(float(ari), 3)})

fixed_df = pd.DataFrame(fixed_rows)
ari_df = pd.DataFrame(ari_rows)
brightness_df = pd.DataFrame(brightness_rows)

print("=== Fixed k=2 clustering ===")
print(fixed_df.to_string(index=False))
print("\n=== Cross-variant agreement (ARI) ===")
print(ari_df.to_string(index=False))
print("\n=== Cluster vs. RAW mean_level ===")
print(brightness_df.to_string(index=False))

save_table(fixed_df, "tbl12_per_region_fixed_k2")
save_table(ari_df, "tbl13_per_region_cross_variant_ari")
save_table(brightness_df, "tbl14_per_region_cluster_brightness")

fig, ax = plt.subplots(figsize=(_FW * 1.2, _FW * 0.55))
pairs = ari_df.apply(lambda r: f"{r['variant_a']}/{r['variant_b']}", axis=1).unique()
x = np.arange(len(REGION_ORDER))
w = 0.8 / len(pairs)
for i, pair in enumerate(pairs):
    vals = []
    for region in REGION_ORDER:
        sub = ari_df[(ari_df["region"] == region) &
                     (ari_df.apply(lambda r: f"{r['variant_a']}/{r['variant_b']}",
                                   axis=1) == pair)]
        vals.append(sub["ari"].iloc[0] if len(sub) else np.nan)
    ax.bar(x + (i - (len(pairs) - 1) / 2) * w, vals, w, label=pair)
ax.set_xticks(x)
ax.set_xticklabels(REGION_ORDER)
ax.set_ylabel("adjusted rand index")
ax.set_ylim(0, 1.05)
ax.set_title("Cross-Variant Cluster Agreement at k=2")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()
save_fig(fig, "fig09_per_region_cross_variant_ari")

In [ ]:
# === Cell 13: Cluster Mean Spectra and Differences ===

DIFF_VARIANT = "snv"          # clustering source
DIFF_TOP_N = 5
# Raw mean spectra (+/- SEM) per cluster; shape = spectrum / its mean, A = larger cluster.

diff_rows = []
fig, axes = plt.subplots(2, len(REGION_ORDER),
                         figsize=(_FW * 0.62 * len(REGION_ORDER), _FW * 0.95),
                         squeeze=False)

for c, region in enumerate(REGION_ORDER):
    mask = (IDX["region"] == region).to_numpy()
    idx_r = IDX[mask].reset_index(drop=True)
    M_raw_r = M_RAW[mask]

    clusters = REGION_CLUSTERS[(region, DIFF_VARIANT)]
    merged = idx_r[["cutout_id"]].merge(clusters, on="cutout_id", how="left")
    lab = merged["cluster"].to_numpy()

    sizes = np.bincount(lab[~pd.isna(lab)].astype(int))
    big, small = (0, 1) if sizes[0] >= sizes[1] else (1, 0)

    M_big = M_raw_r[lab == big]
    M_small = M_raw_r[lab == small]

    mean_big, mean_small = M_big.mean(axis=0), M_small.mean(axis=0)
    sem_big = M_big.std(axis=0) / np.sqrt(M_big.shape[0])
    sem_small = M_small.std(axis=0) / np.sqrt(M_small.shape[0])

    ax = axes[0][c]
    ax.plot(WL, mean_big, color="#1f77b4", label=f"cluster A (n={M_big.shape[0]})")
    ax.fill_between(WL, mean_big - sem_big, mean_big + sem_big, alpha=0.2,
                    color="#1f77b4")
    ax.plot(WL, mean_small, color="#d62728", label=f"cluster B (n={M_small.shape[0]})")
    ax.fill_between(WL, mean_small - sem_small, mean_small + sem_small, alpha=0.2,
                    color="#d62728")
    ax.set_title(region)
    ax.legend(loc="best")
    if c == 0:
        ax.set_ylabel("mean DN (raw)")

    shape_big = mean_big / mean_big.mean()
    shape_small = mean_small / mean_small.mean()
    dshape = shape_big - shape_small

    ax = axes[1][c]
    ax.plot(WL, dshape, color="black", lw=1.2)
    ax.axhline(0, color="grey", lw=0.6)
    ax.set_xlabel("Wavelength (nm)")
    if c == 0:
        ax.set_ylabel("shape-normalised\nA - B")

    order = np.argsort(-np.abs(dshape))[:DIFF_TOP_N]
    diff_rows.append({
        "region": region, "cluster_source_variant": DIFF_VARIANT,
        "n_cluster_a": int(M_big.shape[0]), "n_cluster_b": int(M_small.shape[0]),
        "wl_at_max_abs_diff_nm": round(float(WL[order[0]]), 1),
        "max_abs_shape_diff": round(float(np.abs(dshape[order[0]])), 4),
        "top_diff_wl_nm": ", ".join(f"{WL[i]:.0f}" for i in order),
    })

fig.suptitle(f"Region Cluster Mean Spectra and Shape Difference "
             f"(clusters from {DIFF_VARIANT}, k=2)", y=1.01)
fig.tight_layout()
plt.show()
save_fig(fig, "fig10_region_cluster_mean_spectra_difference")

diff_df = pd.DataFrame(diff_rows)
print(diff_df.to_string(index=False))
save_table(diff_df, "tbl15_region_cluster_shape_difference")

In [ ]:
# === Cell 14: Region Cluster RGB Crops ===

RGB_VARIANT = "snv"
RGB_N_EXAMPLES = 10

TRUE_GT_YOLO_DS = DATA_ROOT / TRUE_GT_RUN_ID / "yolo_dataset"
IMAGE_CANDIDATES = [
    (TRUE_GT_YOLO_DS / "images", "{cid}.jpg"),
    (TRUE_GT_YOLO_DS / "images", "{cid}.png"),
    (TRUE_GT_YOLO_DS / "crops", "{cid}.jpg"),
    (TRUE_GT_YOLO_DS / "crops", "{cid}.png"),
    (DATA_ROOT / TRUE_GT_RUN_ID / "stage2_cutouts" / "images", "{cid}.png"),
    (DATA_ROOT / TRUE_GT_RUN_ID / "stage2_cutouts" / "images", "{cid}.jpg"),
]


def find_object_image(cutout_id):
    """First existing image among IMAGE_CANDIDATES, or None."""
    for directory, pattern in IMAGE_CANDIDATES:
        p = directory / pattern.format(cid=cutout_id)
        if p.exists():
            return p
    return None


def cutout_path_from_geometry(cutout_id):
    """Return the cutout_path from filtered_objects.csv if the file exists, else None."""
    geom_csv = SW_ROOT / "results" / TRUE_GT_RUN_ID / "tables" / "filtered_objects.csv"
    if not geom_csv.exists():
        return None
    g = pd.read_csv(geom_csv)
    if "cutout_path" not in g.columns:
        return None
    hit = g[g["cutout_id"].astype(str) == str(cutout_id)]
    if not len(hit):
        return None
    p = Path(str(hit["cutout_path"].iloc[0]))
    return p if p.exists() else None


def resolve_image(cutout_id):
    return find_object_image(cutout_id) or cutout_path_from_geometry(cutout_id)


probe = REGION_CLUSTERS[("body", RGB_VARIANT)]["cutout_id"].head(3).tolist()
print("probing image locations:")
for cid in probe:
    print(f"  {cid}: {resolve_image(cid)}")

if resolve_image(probe[0]) is None:
    print("\n!! no image found for the probed objects. tried, per cutout_id:")
    for directory, pattern in IMAGE_CANDIDATES:
        print(f"   {directory / pattern.format(cid='<cutout_id>')}")
    print(f"   plus cutout_path from "
          f"{SW_ROOT / 'results' / TRUE_GT_RUN_ID / 'tables' / 'filtered_objects.csv'}")
else:
    rng = np.random.RandomState(RANDOM_SEED)
    for region in REGION_ORDER:
        clusters = REGION_CLUSTERS[(region, RGB_VARIANT)]
        fig, axes = plt.subplots(2, RGB_N_EXAMPLES,
                                 figsize=(1.5 * RGB_N_EXAMPLES, 3.4), squeeze=False)
        for cl in (0, 1):
            cids = clusters.loc[clusters["cluster"] == cl, "cutout_id"].tolist()
            found = [(cid, resolve_image(cid)) for cid in cids]
            found = [(cid, p) for cid, p in found if p is not None]
            if not found:
                continue
            pick = rng.choice(len(found), size=min(RGB_N_EXAMPLES, len(found)),
                              replace=False)
            for col in range(RGB_N_EXAMPLES):
                ax = axes[cl][col]
                ax.set_xticks([]); ax.set_yticks([])
                if col < len(pick):
                    cid, p = found[pick[col]]
                    try:
                        ax.imshow(mpimg.imread(p))
                    except Exception:
                        ax.text(0.5, 0.5, "read\nerror", ha="center", va="center",
                                fontsize=7)
                else:
                    ax.axis("off")
            axes[cl][0].set_ylabel(f"cluster {cl}\n(n={len(cids)})",
                                   fontsize=_FONT_SIZE - 1)
        fig.suptitle(f"{region} - cluster 0 vs. cluster 1 ({RGB_VARIANT}, k=2)")
        fig.tight_layout()
        plt.show()
        save_fig(fig, f"fig11_{region}_cluster_rgb_crops")

In [ ]:
# === Cell 15: Paired Region Comparison per Object ===

PAIR_TOP_N = 6
# Shape-normalised (spectrum / its mean) comparison of two regions of the same object.
PAIRS = [("body", "label"), ("body", "cap")]

pair_tables = {}
for region_a, region_b in PAIRS:
    pos_a = {cid: i for i, cid in enumerate(IDX.loc[IDX["region"] == region_a,
                                                    "cutout_id"])}
    pos_b = {cid: i for i, cid in enumerate(IDX.loc[IDX["region"] == region_b,
                                                    "cutout_id"])}
    M_a = M_RAW[(IDX["region"] == region_a).to_numpy()]
    M_b = M_RAW[(IDX["region"] == region_b).to_numpy()]
    common = sorted(set(pos_a) & set(pos_b))
    print(f"[{region_a}-{region_b}] objects with both: {len(common)}  "
          f"({region_a} only: {len(set(pos_a) - set(pos_b))}, "
          f"{region_b} only: {len(set(pos_b) - set(pos_a))})")

    rows, shapes = [], {}
    for cid in common:
        sa = M_a[pos_a[cid]]
        sb = M_b[pos_b[cid]]
        sa_n, sb_n = sa / sa.mean(), sb / sb.mean()
        shapes[cid] = (sa_n, sb_n)
        rows.append({
            "cutout_id": cid,
            "shape_corr": float(np.corrcoef(sa_n, sb_n)[0, 1]),
            "mean_abs_shape_diff": float(np.mean(np.abs(sa_n - sb_n))),
            "wl_at_max_diff_nm": float(WL[np.argmax(np.abs(sa_n - sb_n))]),
        })

    if not rows:
        print(f"!! no objects carry both {region_a} and {region_b} - skipping.")
        continue
    df = pd.DataFrame(rows).sort_values("shape_corr", ascending=False)
    pair_tables[(region_a, region_b)] = (df, shapes)
    print(df["shape_corr"].describe().to_string())
    save_table(df, f"tbl16_{region_a}_{region_b}_per_object_similarity")

    fig, axes = plt.subplots(1, 3, figsize=(_FW * 1.8, _FW * 0.5))
    axes[0].hist(df["shape_corr"], bins=40, color="#818390")
    axes[0].set_xlabel("shape correlation")
    axes[0].set_ylabel("count")
    axes[0].set_title(f"{region_a} vs {region_b} (n={len(df)})")

    for ax, subset, label in ((axes[1], df.head(PAIR_TOP_N), "most similar"),
                              (axes[2], df.tail(PAIR_TOP_N), "most different")):
        for _, r in subset.iterrows():
            sa_n, sb_n = shapes[r["cutout_id"]]
            ax.plot(WL, sa_n, color=REGION_COLORS[region_a], lw=0.8, alpha=0.7)
            ax.plot(WL, sb_n, color=REGION_COLORS[region_b], lw=0.8, alpha=0.7)
        ax.set_xlabel("Wavelength (nm)")
        ax.set_title(f"{label} (n={len(subset)})")
    axes[1].set_ylabel("shape-normalised value")
    handles = [Line2D([0], [0], color=REGION_COLORS[region_a], label=region_a),
               Line2D([0], [0], color=REGION_COLORS[region_b], label=region_b)]
    axes[1].legend(handles=handles, loc="best")
    fig.tight_layout()
    plt.show()
    save_fig(fig, f"fig12_{region_a}_{region_b}_pair_similarity")

    for subset, label in ((df.head(PAIR_TOP_N), "most_similar"),
                          (df.tail(PAIR_TOP_N), "most_different")):
        paths = [(r["cutout_id"], resolve_image(r["cutout_id"]))
                 for _, r in subset.iterrows()]
        if not any(p for _, p in paths):
            continue
        fig, axes = plt.subplots(1, len(paths), figsize=(2.4 * len(paths), 2.8),
                                 squeeze=False)
        for ax, (cid, p) in zip(axes[0], paths):
            ax.set_xticks([]); ax.set_yticks([])
            corr = float(subset.loc[subset["cutout_id"] == cid, "shape_corr"].iloc[0])
            if p is not None:
                try:
                    ax.imshow(mpimg.imread(p))
                except Exception:
                    ax.text(0.5, 0.5, "read\nerror", ha="center", va="center",
                            fontsize=8)
            else:
                ax.text(0.5, 0.5, "not\nfound", ha="center", va="center", fontsize=8)
            ax.set_title(f"corr={corr:.3f}", fontsize=_FONT_SIZE - 2)
        fig.suptitle(f"{region_a} vs {region_b} - {label.replace('_', ' ')}")
        fig.tight_layout()
        plt.show()
        save_fig(fig, f"fig13_{region_a}_{region_b}_{label}_crops")

In [ ]:
# === Cell 16: Cross-Region Cluster Agreement ===

XREF_VARIANT = "snv"

body_clusters = REGION_CLUSTERS[("body", XREF_VARIANT)].rename(
    columns={"cluster": "body_cluster"})
body_sizes = body_clusters["body_cluster"].value_counts().sort_index()
body_big = int(body_sizes.idxmax())
body_clusters["body_group"] = body_clusters["body_cluster"].map(
    lambda v: "body_A_large" if v == body_big else "body_B_small")

xref_rows = []
for other in ("cap", "label"):
    other_clusters = REGION_CLUSTERS[(other, XREF_VARIANT)].rename(
        columns={"cluster": f"{other}_cluster"})
    merged = body_clusters[["cutout_id", "body_group"]].merge(
        other_clusters, on="cutout_id", how="inner")
    ari = adjusted_rand_score(merged[f"{other}_cluster"], merged["body_group"])
    ct = pd.crosstab(merged[f"{other}_cluster"], merged["body_group"])
    prop = pd.crosstab(merged["body_group"], merged[f"{other}_cluster"],
                       normalize="index")

    print(f"\n=== {other} cluster vs. body group  (n={len(merged)}, ARI={ari:.3f}) ===")
    print(ct.to_string())
    print("\nproportion of each body group per cluster:")
    print(prop.round(3).to_string())

    xref_rows.append({"region": other, "reference": "body", "variant": XREF_VARIANT,
                      "n_matched": len(merged), "ari": round(float(ari), 3)})
    save_table(ct.reset_index(), f"tbl17_{other}_vs_body_crosstab")

xref_df = pd.DataFrame(xref_rows)
print("\n" + xref_df.to_string(index=False))
save_table(xref_df, "tbl18_cross_region_cluster_ari")

fig, ax = plt.subplots(figsize=(_FW * 0.9, _FW * 0.5))
ax.bar(xref_df["region"], xref_df["ari"], color=["#5A56A1", "#972982"], width=0.55)
ax.set_ylabel("adjusted rand index")
ax.set_ylim(0, 1.05)
ax.set_title("Cluster Agreement with Body Cluster")
for xi, v in enumerate(xref_df["ari"]):
    ax.text(xi, v + 0.02, f"{v:.3f}", ha="center", fontsize=_FONT_SIZE - 1)
fig.tight_layout()
plt.show()
save_fig(fig, "fig14_cross_region_cluster_ari")

In [ ]:
# === Cell 17: Pixel-Level Mask Projection ===
# Geometry, alignment and mask projection onto the HSI grid, with the same chain and
# constants as 0802_Generalizability_Spectral_Waste_Preprocessing.ipynb, so the pixels
# match those behind the exported per-region mean spectra.

RGB_NATIVE = (1184, 1200)              # Teledyne DALSA Linea (W, H)
HSI_NATIVE = (640, 600)                # Specim FX17 (W, H)
RGB_CROP_ROWS = 1190
HSI_CROP_COLS = 605
ALIGN_SX = HSI_CROP_COLS / RGB_NATIVE[0]
ALIGN_SY = HSI_NATIVE[1] / RGB_CROP_ROWS
ERODE_PX = 4

SRC_DATA_ROOT = DATA_ROOT / TRUE_GT_RUN_ID
SRC_RESULTS_ROOT = SW_ROOT / "results" / TRUE_GT_RUN_ID
TRUE_GT_LABELS_DIR = SRC_DATA_ROOT / "yolo_dataset" / "labels_true_gt"
FRAME_INDEX_CSV = SRC_RESULTS_ROOT / "tables" / "frame_index.csv"
GEOMETRY_CSV = SRC_RESULTS_ROOT / "tables" / "filtered_objects.csv"
ALIGN_OFFSETS_CSV = (SW_ROOT / "results" / ALIGN_RUN_ID / "tables" /
                     "session_alignment_offsets.csv")


def _as_text(v):
    try:
        if v is None or pd.isna(v):
            return ""
    except (TypeError, ValueError):
        pass
    s = str(v)
    return "" if s in ("nan", "None", "<NA>", "NaT") else s


def load_frame_index():
    """Frame index rows with a hyper_path, one per stem."""
    fi = pd.read_csv(FRAME_INDEX_CSV)
    fi["hyper_path"] = fi["hyper_path"].map(_as_text)
    fi = fi[fi["hyper_path"] != ""].drop_duplicates("stem").reset_index(drop=True)
    return fi


def load_geometry():
    g = pd.read_csv(GEOMETRY_CSV).drop_duplicates("cutout_id").reset_index(drop=True)
    g["cutout_id"] = g["cutout_id"].astype(str)
    for c in ("session", "split"):
        if c not in g.columns:
            g[c] = ""
    return g


def load_align_offsets():
    off = pd.read_csv(ALIGN_OFFSETS_CSV)
    dx = int(off["dx_applied"].iloc[0]) if "dx_applied" in off.columns else 0
    return off, dx


def rgb_to_hsi_xy(xr, yr, dx=0.0, dy=0.0):
    """Map RGB pixel coordinates to HSI pixel coordinates (pixel-centre convention)."""
    xh = (np.asarray(xr, dtype=np.float64) + 0.5) * ALIGN_SX - 0.5 + dx
    yh = (np.asarray(yr, dtype=np.float64) + 0.5) * ALIGN_SY - 0.5 + dy
    return xh, yh


def get_dy(session, off, ):
    """Vertical offset of a session; median offset if the session is not listed."""
    row = off[off["session"].astype(str) == str(session)]
    return float(row["dy"].iloc[0]) if len(row) else float(off["dy"].median())


def load_cube(path, bands=None):
    """HSI cube as (bands, H, W) float32."""
    p = Path(path)
    ext = p.suffix.lower()
    if ext in (".tif", ".tiff"):
        if tifffile is None:
            raise RuntimeError("tifffile not installed")
        arr = tifffile.imread(str(p))
    elif ext == ".hdr":
        if envi is None:
            raise RuntimeError("spectral (ENVI reader) not installed")
        arr = np.transpose(np.asarray(envi.open(str(p)).load()), (2, 0, 1))
    elif ext == ".npy":
        arr = np.load(p)
    elif ext == ".npz":
        with np.load(p) as z:
            arr = z[list(z.files)[0]]
    else:
        raise ValueError(f"unsupported cube format: {p.suffix}")
    arr = np.asarray(arr)
    if arr.ndim != 3:
        raise ValueError(f"expected 3D cube, got {arr.shape} from {p.name}")
    if arr.shape[0] != N_BANDS:
        cand = [i for i, s in enumerate(arr.shape) if s == N_BANDS]
        arr = np.moveaxis(arr, cand[0] if cand else int(np.argmin(arr.shape)), 0)
    if bands is not None:
        arr = arr[bands]
    return arr.astype(np.float32)


def parse_yolo_seg(txt_path, width, height):
    """Rasterise a YOLO-seg polygon file into per-region boolean masks of (height, width)."""
    if _cv2 is None:
        raise RuntimeError("cv2 required to rasterise true-GT polygons")
    acc = {k: np.zeros((int(height), int(width)), dtype=np.uint8) for k in REGION_ORDER}
    for line in Path(txt_path).read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 7:
            continue
        name = CLASS_NAMES.get(int(float(parts[0])))
        if name not in acc:
            continue
        pts = np.asarray([float(v) for v in parts[1:]], dtype=np.float64)
        if pts.size % 2:
            pts = pts[:-1]
        pts = pts.reshape(-1, 2)
        pts[:, 0] *= float(width)
        pts[:, 1] *= float(height)
        _cv2.fillPoly(acc[name], [np.rint(pts).astype(np.int32)], 1)
    return {k: v.astype(bool) for k, v in acc.items()}


def apply_region_priority(masks):
    """body_valid = body AND NOT label AND NOT cap."""
    return {"body": masks["body"] & ~masks["label"] & ~masks["cap"],
            "cap": masks["cap"], "label": masks["label"]}


def cutout_mask_to_hsi(mask_cut, row, dx, dy, hsi_shape, erode_px=ERODE_PX):
    """Project an eroded cutout-space mask onto the raw HSI grid."""
    if erode_px > 0 and mask_cut.any():
        iters = max(1, int(round(float(erode_px) / ALIGN_SX)))
        eroded = ndimage.binary_erosion(mask_cut, iterations=iters)
        if eroded.any():
            mask_cut = eroded
    ys, xs = np.nonzero(mask_cut)
    out = np.zeros(hsi_shape, dtype=bool)
    if xs.size == 0:
        return out
    x_rgb = xs - int(row["canvas_offset_x"]) + int(row["mask_bbox_x1"])
    y_rgb = ys - int(row["canvas_offset_y"]) + int(row["mask_bbox_y1"])
    keep = y_rgb < RGB_CROP_ROWS
    x_rgb, y_rgb = x_rgb[keep], y_rgb[keep]
    if x_rgb.size == 0:
        return out
    hx, hy = rgb_to_hsi_xy(x_rgb, y_rgb, dx, dy)
    hx = np.rint(hx).astype(np.int64)
    hy = np.rint(hy).astype(np.int64)
    ok = (hx >= 0) & (hx < hsi_shape[1]) & (hy >= 0) & (hy < hsi_shape[0])
    out[hy[ok], hx[ok]] = True
    return out


print(f"labels : {TRUE_GT_LABELS_DIR}  exists={TRUE_GT_LABELS_DIR.is_dir()}")
print(f"frames : {FRAME_INDEX_CSV}  exists={FRAME_INDEX_CSV.exists()}")
print(f"geom   : {GEOMETRY_CSV}  exists={GEOMETRY_CSV.exists()}")
print(f"align  : {ALIGN_OFFSETS_CSV}  exists={ALIGN_OFFSETS_CSV.exists()}")
print(f"scale x {ALIGN_SX:.7f}   scale y {ALIGN_SY:.7f}   erode {ERODE_PX} hsi px")

In [ ]:
# === Cell 18: Within- vs. Between-Object Variance ===
# Per-pixel spectra are recovered by re-reading the HSI cubes and re-projecting the
# true-GT masks for a random object sample.

PIXEL_N_OBJECTS = 120          # objects sampled for the within-object statistics
PIXEL_MIN_PX = 5

_ready = (TRUE_GT_LABELS_DIR.is_dir() and FRAME_INDEX_CSV.exists()
          and GEOMETRY_CSV.exists() and ALIGN_OFFSETS_CSV.exists() and _cv2 is not None)

if not _ready:
    print("!! prerequisites missing - see the paths printed by Cell 17.")
else:
    frame_index = load_frame_index()
    geometry = load_geometry()
    align_off, align_dx = load_align_offsets()
    label_paths = {p.stem: p for p in sorted(TRUE_GT_LABELS_DIR.glob("*.txt"))}

    pool = geometry[geometry["cutout_id"].isin(set(label_paths))].copy()
    pool = pool[pool["cutout_id"].isin(set(IDX["cutout_id"]))].reset_index(drop=True)
    print(f"objects with labels, geometry and exported spectra: {len(pool):,}")

    rng = np.random.RandomState(RANDOM_SEED)
    take = rng.choice(len(pool), size=min(PIXEL_N_OBJECTS, len(pool)), replace=False)
    sample = pool.iloc[np.sort(take)].reset_index(drop=True)

    frame_lookup = dict(zip(frame_index["stem"].map(_as_text),
                            frame_index["hyper_path"].map(_as_text)))

    within_rows, failures = [], []
    cube_cache_stem, cube_cache = None, None

    for _, row in sample.iterrows():
        cid, stem = str(row["cutout_id"]), _as_text(row["stem"])
        hyper = frame_lookup.get(stem, "")
        if not hyper:
            failures.append(f"{cid}: no hyper_path for stem {stem}")
            continue
        try:
            if stem != cube_cache_stem:
                cube_cache = load_cube(hyper)
                cube_cache_stem = stem
            cube = cube_cache
            hsi_shape = cube.shape[1:]
            dy = get_dy(_as_text(row.get("session")), align_off)

            masks = apply_region_priority(
                parse_yolo_seg(label_paths[cid], int(row["canvas_w"]),
                               int(row["canvas_h"])))

            for region in REGION_ORDER:
                mc = masks[region]
                if not mc.any():
                    continue
                hm = cutout_mask_to_hsi(mc, row, align_dx, dy, hsi_shape)
                n_px = int(hm.sum())
                if n_px < PIXEL_MIN_PX:
                    continue
                px = cube[:, hm].astype(np.float64)            # (bands, n_px)
                mean_spec = px.mean(axis=1)
                px_shape = px / (px.mean(axis=0, keepdims=True) + 1e-9)
                mean_shape = mean_spec / mean_spec.mean()

                within_rows.append({
                    "cutout_id": cid, "region": region, "n_px": n_px,
                    "within_var_dn": float(np.mean(np.var(px, axis=1))),
                    "within_var_shape": float(np.mean(np.var(px_shape, axis=1))),
                    "mean_level": float(mean_spec.mean()),
                    "mean_shape_sd": float(np.mean(np.std(px_shape, axis=1))),
                    "median_px_corr_to_mean": float(np.median([
                        np.corrcoef(px_shape[:, j], mean_shape)[0, 1]
                        for j in rng.choice(n_px, size=min(40, n_px), replace=False)])),
                })
        except Exception as e:
            failures.append(f"{cid}: {type(e).__name__}: {e}")

within_df = pd.DataFrame(within_rows)

print(f"sample size: {len(sample)}   rows extracted: {len(within_rows)}   "
      f"failures: {len(failures)}")
if failures:
    print("failure examples:")
    for f in failures[:10]:
        print(f"  {f}")

if within_df.empty:
    print("\n!! within_df is empty - nothing to aggregate. Likely causes:")
    print("   - every sampled object raised an exception during extraction (see")
    print("     failures above)")
    print("   - every region's HSI pixel count fell below PIXEL_MIN_PX "
          f"({PIXEL_MIN_PX})")
    print("   - the geometry/frame_index/labels join produced an empty pool")
    print(f"   pool size: {len(pool) if 'pool' in dir() else 'undefined'}")
    print(f"   sample size: {len(sample) if 'sample' in dir() else 'undefined'}")
else:
    between_rows = []
    for region in REGION_ORDER:
        mask = (IDX["region"] == region).to_numpy()
        M_r = M_RAW[mask]
        shape_r = M_r / M_r.mean(axis=1, keepdims=True)
        between_rows.append({
            "region": region, "n_objects": int(M_r.shape[0]),
            "between_var_dn": float(np.mean(np.var(M_r, axis=0))),
            "between_var_shape": float(np.mean(np.var(shape_r, axis=0))),
        })
    between_df = pd.DataFrame(between_rows)

    agg = within_df.groupby("region").agg(
        n_objects_sampled=("cutout_id", "nunique"),
        median_n_px=("n_px", "median"),
        within_var_dn=("within_var_dn", "mean"),
        within_var_shape=("within_var_shape", "mean"),
        median_px_corr_to_mean=("median_px_corr_to_mean", "median"),
    ).reset_index()

    variance_df = agg.merge(between_df, on="region")
    variance_df["within_to_between_dn"] = (
        variance_df["within_var_dn"] / variance_df["between_var_dn"]).round(3)
    variance_df["within_to_between_shape"] = (
        variance_df["within_var_shape"] / variance_df["between_var_shape"]).round(3)

    print("\n=== Within-object vs. between-object variance ===")
    print(variance_df.to_string(index=False))
    save_table(within_df, "tbl19_within_object_pixel_variance_per_object")
    save_table(variance_df, "tbl20_within_vs_between_variance")

    fig, axes = plt.subplots(1, 3, figsize=(_FW * 1.8, _FW * 0.5))

    ax = axes[0]
    x = np.arange(len(variance_df))
    w = 0.38
    ax.bar(x - w / 2, variance_df["within_to_between_dn"], w, label="raw DN",
           color="#5A56A1")
    ax.bar(x + w / 2, variance_df["within_to_between_shape"], w,
           label="shape-normalised", color="#88BEE7")
    ax.axhline(1.0, color="grey", ls="--", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(variance_df["region"])
    ax.set_ylabel("within / between variance")
    ax.set_title("Aggregation Ceiling")
    ax.legend(loc="best")

    ax = axes[1]
    for region in REGION_ORDER:
        sub = within_df[within_df["region"] == region]
        ax.hist(sub["median_px_corr_to_mean"], bins=25, alpha=0.55,
                color=REGION_COLORS[region], label=region)
    ax.set_xlabel("median pixel-to-mean shape correlation")
    ax.set_ylabel("count")
    ax.set_title("Within-Object Homogeneity")
    ax.legend(loc="best")

    ax = axes[2]
    for region in REGION_ORDER:
        sub = within_df[within_df["region"] == region]
        ax.scatter(sub["n_px"], sub["within_var_shape"], s=10, alpha=0.6,
                   color=REGION_COLORS[region], label=region)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("region pixels (HSI grid)")
    ax.set_ylabel("within-object shape variance")
    ax.set_title("Variance vs. Region Size")
    ax.legend(loc="best")

    fig.tight_layout()
    plt.show()
    save_fig(fig, "fig15_within_vs_between_object_variance")

In [ ]:
# === Cell 19: Per-Pixel Spectra Examples ===

PIXEL_PREVIEW_N = 3
PIXEL_PREVIEW_MAX_TRACES = 150

if not _ready or within_df.empty:
    print("!! Cell 18 produced no per-pixel data - skipping.")
else:
    counts = within_df.pivot_table(index="cutout_id", columns="region",
                                   values="n_px", aggfunc="first")
    counts = counts.dropna()
    if counts.empty:
        print("!! no object has all three regions in the sampled set.")
    else:
        rng = np.random.RandomState(RANDOM_SEED)
        pick_ids = counts.sample(n=min(PIXEL_PREVIEW_N, len(counts)),
                                 random_state=RANDOM_SEED).index.tolist()

        frame_lookup = dict(zip(frame_index["stem"].map(_as_text),
                                frame_index["hyper_path"].map(_as_text)))
        geo_lookup = geometry.set_index("cutout_id")

        fig, axes = plt.subplots(len(pick_ids), len(REGION_ORDER),
                                 figsize=(_FW * 0.62 * len(REGION_ORDER),
                                          _FW * 0.45 * len(pick_ids)),
                                 squeeze=False, sharex=True)

        for r, cid in enumerate(pick_ids):
            row = geo_lookup.loc[cid]
            cube = load_cube(frame_lookup[_as_text(row["stem"])])
            hsi_shape = cube.shape[1:]
            dy = get_dy(_as_text(row.get("session")), align_off)
            masks = apply_region_priority(
                parse_yolo_seg(label_paths[cid], int(row["canvas_w"]),
                               int(row["canvas_h"])))

            for c, region in enumerate(REGION_ORDER):
                ax = axes[r][c]
                mc = masks[region]
                if not mc.any():
                    ax.set_axis_off()
                    continue
                hm = cutout_mask_to_hsi(mc, row, align_dx, dy, hsi_shape)
                n_px = int(hm.sum())
                if n_px < 1:
                    ax.set_axis_off()
                    continue
                px = cube[:, hm].astype(np.float64)
                px_shape = px / (px.mean(axis=0, keepdims=True) + 1e-9)
                sel = rng.choice(n_px, size=min(PIXEL_PREVIEW_MAX_TRACES, n_px),
                                 replace=False)
                for j in sel:
                    ax.plot(WL, px_shape[:, j], color=REGION_COLORS[region],
                            lw=0.3, alpha=0.15)
                mean_spec = px.mean(axis=1)
                ax.plot(WL, mean_spec / mean_spec.mean(), color="black", lw=1.4)
                if r == 0:
                    ax.set_title(region)
                if c == 0:
                    ax.set_ylabel(f"{cid[-18:]}\nshape-normalised")
                ax.text(0.02, 0.98, f"{n_px} px", va="top", ha="left",
                        transform=ax.transAxes, fontsize=_FONT_SIZE - 3)

        for c in range(len(REGION_ORDER)):
            axes[-1][c].set_xlabel("Wavelength (nm)")
        fig.suptitle("Per-Pixel Spectra and Region Mean", y=1.01)
        fig.tight_layout()
        plt.show()
        save_fig(fig, "fig16_per_pixel_spectra_examples")

In [ ]:
# === Cell 20: Output Inventory ===

files = sorted(OUT_DIR.glob("*"))
inv = pd.DataFrame([{
    "file": p.name,
    "kind": p.suffix.lstrip("."),
    "size_kb": round(p.stat().st_size / 1024, 1),
} for p in files if p.is_file()])

print(f"{OUT_DIR}\n")
print(inv.to_string(index=False))
print(f"\n{len(inv)} files   "
      f"{inv[inv['kind'] == 'png'].shape[0]} figures   "
      f"{inv[inv['kind'] == 'csv'].shape[0]} tables")

inv.to_csv(OUT_DIR / "tbl21_output_inventory.csv", index=False)